# Lab 052 — Streaming Agent Responses

## Lab Intro

In previous labs, our agents executed like this:

```text
User Request
      ↓
Agent Runs
      ↓
Tool Executes
      ↓
Final Response Returned
```

The user only sees the result after the entire workflow completes.

For simple tasks this is fine.

However, many agent workflows involve:

- multiple tool calls
- API requests
- database queries
- document processing
- long-running reasoning

Waiting silently for several seconds creates a poor user experience.

To solve this, LangGraph supports:

> **Streaming**

Streaming allows us to observe graph execution in real time as nodes complete and state changes occur.

---

## Key Idea

Without streaming:

```text
Run Entire Graph
      ↓
Return Final Result
```

With streaming:

```text
Node Executes
      ↓
State Update Appears
      ↓
Next Node Executes
      ↓
New Update Appears
      ↓
Continue Until Completion
```

Users can see progress while the workflow is running.

---

## Enterprise Analogy

Imagine ordering food at a restaurant.

Without streaming:

```text
Order Placed
(wait...)
Food Arrives
```

You have no visibility into what is happening.

With streaming:

```text
Order Received
↓
Preparing Food
↓
Cooking
↓
Ready
↓
Delivered
```

The result is the same, but the experience is significantly better.

Streaming provides similar visibility for AI agents.

---

## Lab Code

In [ ]:
from typing import Annotated

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from langgraph.graph import (
    StateGraph,
    START,
    END
)

from langgraph.graph.message import (
    add_messages
)

from langgraph.prebuilt import ToolNode

from typing_extensions import TypedDict


# -------------------------
# Tool
# -------------------------

@tool
def calculator(expression: str) -> str:
    """
    Evaluate a mathematical expression.
    """
    return str(eval(expression))


# -------------------------
# State
# -------------------------

class State(TypedDict):
    messages: Annotated[list, add_messages]


# -------------------------
# LLM
# -------------------------

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

tools = [calculator]

llm_with_tools = llm.bind_tools(
    tools
)


# -------------------------
# Chatbot Node
# -------------------------

def chatbot(state: State):

    response = llm_with_tools.invoke(
        state["messages"]
    )

    return {
        "messages": [response]
    }


# -------------------------
# Tool Node
# -------------------------

tool_node = ToolNode(tools)


# -------------------------
# Router
# -------------------------

def route_tools(state: State):

    last_message = state["messages"][-1]

    if last_message.tool_calls:
        return "tools"

    return END


# -------------------------
# Build Graph
# -------------------------

builder = StateGraph(State)

builder.add_node(
    "chatbot",
    chatbot
)

builder.add_node(
    "tools",
    tool_node
)

builder.add_edge(
    START,
    "chatbot"
)

builder.add_conditional_edges(
    "chatbot",
    route_tools
)

builder.add_edge(
    "tools",
    "chatbot"
)

graph = builder.compile()


# -------------------------
# Stream Execution
# -------------------------

for event in graph.stream(
    {
        "messages": [
            (
                "user",
                "What is 125 * 8?"
            )
        ]
    }
):
    print(event)

---

## Example Output

```python
{
    'chatbot': {
        'messages': [
            AIMessage(
                content='',
                tool_calls=[...]
            )
        ]
    }
}

{
    'tools': {
        'messages': [
            ToolMessage(
                content='1000'
            )
        ]
    }
}

{
    'chatbot': {
        'messages': [
            AIMessage(
                content=
                '125 multiplied by 8 equals 1000.'
            )
        ]
    }
}
```

---

## Explanation

## What Is Streaming?

Streaming allows us to receive updates while the graph is executing.

Instead of:

```python
result = graph.invoke(...)
```

which waits for completion,

we use:

```python
graph.stream(...)
```

which produces events incrementally.

---

## Step 1 — Start Execution

The graph receives:

```text
What is 125 * 8?
```

Execution begins.

Instead of waiting for completion:

```text
Updates arrive immediately.
```

---

## Step 2 — First Event

The chatbot node executes.

Output:

```python
{
    "chatbot": {
        "messages": [...]
    }
}
```

The event contains:

```text
Tool call request
```

generated by the LLM.

At this point:

```text
Tool execution has not happened yet.
```

---

## Step 3 — Second Event

The ToolNode executes.

Output:

```python
{
    "tools": {
        "messages": [...]
    }
}
```

The tool result appears:

```text
1000
```

The graph continues.

---

## Step 4 — Third Event

The chatbot receives:

```text
Tool observation
```

and generates the final answer.

Output:

```python
{
    "chatbot": {
        "messages": [...]
    }
}
```

The workflow completes.

---

## Execution Timeline

Without streaming:

```text
Time 0s  → Start
Time 3s  → Final Answer
```

Nothing is visible in between.

---

With streaming:

```text
Time 0s → Chatbot Event

Time 1s → Tool Event

Time 2s → Final Response Event
```

Progress becomes observable.

---

## Why Streaming Matters

### Better User Experience

Instead of:

```text
Waiting...
```

users see activity immediately.

---

### Long-Running Tasks

Useful for:

```text
Research agents
Document analysis
Multi-step workflows
Data processing
```

where execution may take several seconds or minutes.

---

### Debugging

Streaming reveals:

```text
Node execution order
```

and

```text
Intermediate state updates
```

This makes troubleshooting significantly easier.

---

### Monitoring

Production systems often stream events to:

```text
Dashboards
Logs
Tracing systems
Observability platforms
```

for operational visibility.

---

## Stream Event Structure

Each event typically contains:

```python
{
    "node_name": {
        ...
    }
}
```

Example:

```python
{
    "tools": {
        "messages": [...]
    }
}
```

This indicates:

```text
Tool node completed execution.
```

---

## Streaming vs Invoke

### invoke()

```python
result = graph.invoke(...)
```

Behavior:

```text
Run entire graph
Return final state
```

---

### stream()

```python
for event in graph.stream(...):
```

Behavior:

```text
Run graph incrementally
Return updates as they occur
```

---

## Common Streaming Pattern

A common production implementation:

```python
for event in graph.stream(input):

    print(
        "NODE:",
        list(event.keys())[0]
    )
```

Output:

```text
NODE: chatbot
NODE: tools
NODE: chatbot
```

This provides a simple execution trace.

---

## Streaming Agent Lifecycle

```text
User Request
      │
      ▼
Chatbot Event
      │
      ▼
Tool Event
      │
      ▼
Chatbot Event
      │
      ▼
END
```

Every step becomes visible.

---

## Beyond State Streaming

LangGraph supports additional streaming modes.

Examples include:

### State Updates

```text
Node outputs
```

---

### Message Streaming

```text
LLM token generation
```

---

### Event Streaming

```text
Execution events
```

---

### Custom Streaming

```text
Application-defined updates
```

These capabilities enable highly interactive agent experiences.

---

## Common Mistakes

### 1. Expecting a Final Result Immediately

With:

```python
graph.stream(...)
```

you receive:

```text
Multiple events
```

not a single final object.

---

### 2. Ignoring Intermediate Events

Many developers only inspect:

```text
Final response
```

and miss valuable debugging information.

Streaming exposes the entire workflow.

---

### 3. Using invoke() for Long Tasks

For lengthy workflows:

```python
graph.invoke(...)
```

can feel unresponsive.

Streaming usually provides a better experience.

---

## Mental Model

Think of streaming as:

```text
Watching the workflow happen
```

instead of:

```text
Waiting for the workflow to finish
```

You observe execution step by step.

---

## Architecture

```text
               User Request
                      │
                      ▼
                 Graph Start
                      │
                      ▼
               Stream Event #1
                (Chatbot)
                      │
                      ▼
               Stream Event #2
                 (ToolNode)
                      │
                      ▼
               Stream Event #3
                (Chatbot)
                      │
                      ▼
                    END
```

Each completed step generates an event.

---

## Key Takeaways

- Streaming allows graph execution updates to be observed in real time.
- `graph.stream()` emits events as nodes complete.
- Streaming improves user experience for long-running workflows.
- It provides visibility into tool execution and reasoning loops.
- Streaming is extremely useful for debugging and monitoring.
- Each event typically represents the output of a completed node.
- Production agent systems frequently rely on streaming for responsiveness and observability.
- In the next lab, we will learn how to generate structured outputs from agents and enforce predictable response formats.